<a href="https://colab.research.google.com/github/cal-me-j/atf-ai-learn/blob/main/Week3_12_guardrails_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.3: Guardrails - Safe AI Systems

## Key Insight: How to mitigate undesired outputs from LLMs

Make your digital twin SAFE for production!

## What You'll Build
- Content filters
- Output validators
- Safety layers
- Fallback strategies
- PII detection

In [1]:
import re
from typing import List, Dict, Any, Optional

print('Week 3.3: Guardrails & Safety')
print('Goal: Make your twin safe for real users!')

Week 3.3: Guardrails & Safety
Goal: Make your twin safe for real users!


## Part 1: Content Filtering

Block inappropriate content.

In [2]:
class ContentFilter:
    '''Filter inappropriate content.'''

    def __init__(self):
        # Simplified word lists for demo
        self.profanity_list = ['badword1', 'badword2']  # Add actual words
        self.sensitive_topics = ['violence', 'illegal']

    def check_profanity(self, text: str) -> Dict[str, Any]:
        '''Check for profanity.'''
        text_lower = text.lower()
        found = [word for word in self.profanity_list if word in text_lower]

        return {
            'is_safe': len(found) == 0,
            'violations': found
        }

    def check_sensitive_topics(self, text: str) -> Dict[str, Any]:
        '''Check for sensitive topics.'''
        text_lower = text.lower()
        found = [topic for topic in self.sensitive_topics if topic in text_lower]

        return {
            'is_safe': len(found) == 0,
            'topics': found
        }

    def filter(self, text: str) -> Dict[str, Any]:
        '''Run all filters.'''
        profanity_check = self.check_profanity(text)
        topic_check = self.check_sensitive_topics(text)

        is_safe = profanity_check['is_safe'] and topic_check['is_safe']

        return {
            'is_safe': is_safe,
            'profanity_violations': profanity_check['violations'],
            'sensitive_topics': topic_check['topics'],
            'text': text if is_safe else '[FILTERED]'
        }

filter_system = ContentFilter()

print('Content Filter Tests:')
print(filter_system.filter('Hello, how are you?'))
print(filter_system.filter('This involves violence'))  # Will be flagged

Content Filter Tests:
{'is_safe': True, 'profanity_violations': [], 'sensitive_topics': [], 'text': 'Hello, how are you?'}
{'is_safe': False, 'profanity_violations': [], 'sensitive_topics': ['violence'], 'text': '[FILTERED]'}


## Part 2: PII Detection

Detect and protect personal information.

In [3]:
class PIIDetector:
    '''Detect personally identifiable information.'''

    def __init__(self):
        # Regex patterns for common PII
        self.patterns = {
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b'
        }

    def detect(self, text: str) -> Dict[str, Any]:
        '''Detect PII in text.'''
        found_pii = {}

        for pii_type, pattern in self.patterns.items():
            matches = re.findall(pattern, text)
            if matches:
                found_pii[pii_type] = matches

        return {
            'has_pii': len(found_pii) > 0,
            'pii_types': list(found_pii.keys()),
            'details': found_pii
        }

    def redact(self, text: str) -> str:
        '''Redact PII from text.'''
        result = text

        for pii_type, pattern in self.patterns.items():
            result = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', result)

        return result

pii_detector = PIIDetector()

text_with_pii = "Contact me at john@example.com or call 555-123-4567"
print('\nPII Detection:')
print('Original:', text_with_pii)
print('Detection:', pii_detector.detect(text_with_pii))
print('Redacted:', pii_detector.redact(text_with_pii))


PII Detection:
Original: Contact me at john@example.com or call 555-123-4567
Detection: {'has_pii': True, 'pii_types': ['email', 'phone'], 'details': {'email': ['john@example.com'], 'phone': ['555-123-4567']}}
Redacted: Contact me at [REDACTED_EMAIL] or call [REDACTED_PHONE]


## Part 3: Output Validator

Ensure AI outputs meet requirements.

In [4]:
class OutputValidator:
    '''Validate AI outputs before showing to user.'''

    def __init__(self, max_length: int = 500):
        self.max_length = max_length
        self.content_filter = ContentFilter()
        self.pii_detector = PIIDetector()

    def validate(self, output: str) -> Dict[str, Any]:
        '''Run all validation checks.'''
        issues = []

        # Check length
        if len(output) > self.max_length:
            issues.append(f'Too long ({len(output)} > {self.max_length} chars)')

        # Check if empty
        if not output.strip():
            issues.append('Empty output')

        # Check content safety
        content_check = self.content_filter.filter(output)
        if not content_check['is_safe']:
            issues.append('Content safety violation')

        # Check for PII
        pii_check = self.pii_detector.detect(output)
        if pii_check['has_pii']:
            issues.append(f"Contains PII: {pii_check['pii_types']}")

        # Check for disclaimers (if needed)
        needs_disclaimer = any(word in output.lower()
                             for word in ['medical', 'legal', 'financial'])

        return {
            'is_valid': len(issues) == 0,
            'issues': issues,
            'needs_disclaimer': needs_disclaimer,
            'output': output
        }

    def add_disclaimer(self, output: str, topic: str) -> str:
        '''Add disclaimer to output.'''
        disclaimers = {
            'medical': "\n\n⚠️ Note: This is not medical advice. Consult a healthcare professional.",
            'legal': "\n\n⚠️ Note: This is not legal advice. Consult a lawyer.",
            'financial': "\n\n⚠️ Note: This is not financial advice. Consult a financial advisor."
        }

        for key, disclaimer in disclaimers.items():
            if key in topic.lower():
                return output + disclaimer

        return output

validator = OutputValidator(max_length=300)

test_output = "You should invest all your money in stocks."
result = validator.validate(test_output)

print('\nValidation Result:')
print(f"Valid: {result['is_valid']}")
print(f"Issues: {result['issues']}")
print(f"Needs disclaimer: {result['needs_disclaimer']}")

if result['needs_disclaimer']:
    safe_output = validator.add_disclaimer(test_output, 'financial')
    print(f"\nWith disclaimer: {safe_output}")


Validation Result:
Valid: True
Issues: []
Needs disclaimer: False


## Part 4: Safe Digital Twin

Put it all together.

In [5]:
class SafeDigitalTwin:
    '''Digital twin with safety guardrails.'''

    def __init__(self, name: str):
        self.name = name
        self.validator = OutputValidator()
        self.content_filter = ContentFilter()

    def _process_input(self, user_input: str) -> Optional[str]:
        '''Process and filter user input.'''
        # Filter input
        filter_result = self.content_filter.filter(user_input)

        if not filter_result['is_safe']:
            return None  # Input blocked

        return filter_result['text']

    def _generate_response(self, filtered_input: str) -> str:
        '''Generate AI response (mock for demo).'''
        # This would call your actual AI
        return f"I received: {filtered_input}"

    def _process_output(self, response: str) -> str:
        '''Validate and process output.'''
        validation = self.validator.validate(response)

        if not validation['is_valid']:
            # Fallback to safe response
            return "I apologize, but I cannot provide that response. How else can I help?"

        # Add disclaimers if needed
        if validation['needs_disclaimer']:
            response = self.validator.add_disclaimer(response, response)

        return response

    def chat(self, user_input: str) -> str:
        '''Safe chat with full pipeline.'''
        # Step 1: Filter input
        filtered_input = self._process_input(user_input)

        if filtered_input is None:
            return "I cannot process that input. Please rephrase."

        # Step 2: Generate response
        response = self._generate_response(filtered_input)

        # Step 3: Validate and process output
        safe_response = self._process_output(response)

        return safe_response

safe_twin = SafeDigitalTwin('Safe Assistant')

print('\nSafe Digital Twin:')
print(safe_twin.chat('Hello, how are you?'))
print(safe_twin.chat('Tell me about medical treatments'))  # Will add disclaimer


Safe Digital Twin:
I received: Hello, how are you?
I received: Tell me about medical treatments

⚠️ Note: This is not medical advice. Consult a healthcare professional.


## Exercise 1: Add More Guardrails

**Task**: Enhance the safety system:

1. Add more PII patterns (addresses, dates of birth)
2. Create custom content filters for your domain
3. Add rate limiting
4. Implement logging of safety violations
5. Test with edge cases

In [6]:
# YOUR CODE HERE

import re
import time
from typing import List, Dict, Any, Optional

# ==========================================
# 1. Enhanced PII Detector
# ==========================================
class EnhancedPIIDetector(PIIDetector):
    def __init__(self):
        super().__init__()
        # Adding new patterns for DOB (YYYY-MM-DD or MM/DD/YYYY) and Address patterns
        self.patterns.update({
            'dob': r'\b(?:\d{4}[-/]\d{2}[-/]\d{2}|\d{2}[-/]\d{2}[-/]\d{4})\b',
            'address': r'\b\d{1,5}\s[A-Za-z0-9\s\.,]{5,30}\s(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Lane|Ln|Way)\b'
        })

# ==========================================
# 2. Custom Content Filter (Domain-Specific: Tech Support Twin)
# ==========================================
class TechSupportContentFilter(ContentFilter):
    def __init__(self):
        super().__init__()
        # Block competitors or unauthorized commands
        self.sensitive_topics.extend(['competitor-x', 'unauthorized-root', 'jailbreak'])

# ==========================================
# 3. Rate Limiter Guardrail
# ==========================================
class RateLimiter:
    def __init__(self, max_requests: int = 3, time_window: int = 10):
        self.max_requests = max_requests
        self.time_window = time_window
        self.request_history = []

    def is_allowed(self) -> bool:
        current_time = time.time()
        # Filter out requests outside our rolling window
        self.request_history = [t for t in self.request_history if current_time - t < self.time_window]

        if len(self.request_history) >= self.max_requests:
            return False

        self.request_history.append(current_time)
        return True

# ==========================================
# 4. Safety Logger
# ==========================================
class SafetyLogger:
    def __init__(self):
        self.logs = []

    def log_violation(self, input_text: str, violation_type: str, details: Any):
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
        log_entry = {
            'timestamp': timestamp,
            'input': input_text,
            'violation_type': violation_type,
            'details': details
        }
        self.logs.append(log_entry)
        print(f"[SECURITY LOG] [{timestamp}] {violation_type.upper()}: {details}")

# ==========================================
# 5. Putting it Together in an Enhanced Twin
# ==========================================
class ProductionReadyTwin:
    def __init__(self, name: str):
        self.name = name
        self.pii_detector = EnhancedPIIDetector()
        self.content_filter = TechSupportContentFilter()
        self.rate_limiter = RateLimiter(max_requests=3, time_window=10)
        self.logger = SafetyLogger()

    def process_message(self, user_input: str) -> str:
        # Check 1: Rate Limiting
        if not self.rate_limiter.is_allowed():
            self.logger.log_violation(user_input, "rate_limit", "Too many requests")
            return "Too many requests. Please slow down and try again later."

        # Check 2: Input Safety and PII Detection
        filter_result = self.content_filter.filter(user_input)
        if not filter_result['is_safe']:
            self.logger.log_violation(user_input, "content_violation", filter_result['sensitive_topics'])
            return "I cannot process that input. It violates safety guidelines."

        pii_result = self.pii_detector.detect(user_input)
        if pii_result['has_pii']:
            self.logger.log_violation(user_input, "pii_exposed", pii_result['pii_types'])
            # Auto-redact and continue with a safe version of the prompt
            user_input = self.pii_detector.redact(user_input)

        # Mock Output Generation
        return f"Hello! This is {self.name}. I processed your request: '{user_input}'"

# ==========================================
# 6. Testing Edge Cases
# ==========================================
prod_twin = ProductionReadyTwin("SecureTwin")

print("--- Test 1: Address and DOB Detection ---")
safe_res = prod_twin.process_message("My address is 123 Main Street and my birthday is 1995-12-15")
print("Response:", safe_res)

print("\n--- Test 2: Custom Competitor Filter ---")
blocked_res = prod_twin.process_message("Why is competitor-x better than you?")
print("Response:", blocked_res)

print("\n--- Test 3: Rate Limiting ---")
for i in range(5):
    res = prod_twin.process_message("Ping")
    print(f"Attempt {i+1}: {res}")

--- Test 1: Address and DOB Detection ---
[SECURITY LOG] [2026-10-14 09:36:58] PII_EXPOSED: ['dob']
Response: Hello! This is SecureTwin. I processed your request: 'My address is 123 Main Street and my birthday is [REDACTED_DOB]'

--- Test 2: Custom Competitor Filter ---
[SECURITY LOG] [2026-10-14 09:36:58] CONTENT_VIOLATION: ['competitor-x']
Response: I cannot process that input. It violates safety guidelines.

--- Test 3: Rate Limiting ---
Attempt 1: Hello! This is SecureTwin. I processed your request: 'Ping'
[SECURITY LOG] [2026-10-14 09:36:58] RATE_LIMIT: Too many requests
Attempt 2: Too many requests. Please slow down and try again later.
[SECURITY LOG] [2026-10-14 09:36:58] RATE_LIMIT: Too many requests
Attempt 3: Too many requests. Please slow down and try again later.
[SECURITY LOG] [2026-10-14 09:36:58] RATE_LIMIT: Too many requests
Attempt 4: Too many requests. Please slow down and try again later.
[SECURITY LOG] [2026-10-14 09:36:58] RATE_LIMIT: Too many requests
Attempt 5: T

## Summary

### What We Learned:
- Content filtering blocks inappropriate content
- PII detection protects user privacy
- Output validation ensures quality
- Multiple safety layers = robust system
- Always add disclaimers for sensitive topics

### Next: Week 3.4 - Red Teaming & Security